# Flight/Travel Volume vs. Airline Stocks: A Real-Time Demand Nowcast

**Data source**: TSA publishes the actual daily count of travelers screened at U.S. airport
checkpoints — a genuine, real-time read on air-travel demand, released free with no API key at
[tsa.gov/travel/passenger-volumes](https://www.tsa.gov/travel/passenger-volumes) (current year) plus
per-year archive pages back to 2019. This is not a novelty dataset: TSA checkpoint throughput was
widely tracked by hedge funds and sell-side desks in 2020-2021 as a leading/coincident indicator of
air-travel demand recovery, well before airlines' own quarterly guidance confirmed it — headlines like
"TSA volumes are now at 85% of 2019 levels" were literally a trading signal for that period.

**Hypotheses:**
1. TSA checkpoint growth (especially the COVID crash and recovery — a rare, clean natural experiment)
   correlates with airline stock returns, and the two aren't perfectly synchronized — either the
   travel data leads the stock (real demand shows up before guidance does) or the stock leads the
   data (forward-looking markets price in a recovery before it shows up in daily throughput).
2. The relationship should be **specific to airlines**, not just "the market" — comparing against
   the S&P 500 tells us whether TSA data is a genuine sector-specific signal or just another proxy
   for general risk sentiment.
3. **Acceleration/deceleration in the pace of travel-demand recovery** (not the raw level) is what
   should carry signal for forward airline stock returns — an event-study analog to the "attention
   shock" work in the companion Wikipedia notebook.

**Tickers**: **DAL** (Delta), **UAL** (United), **AAL** (American), **LUV** (Southwest) — the four
largest U.S. carriers — plus **JETS** (the US Global Jets ETF, a sector-aggregate proxy) and **^GSPC**
(S&P 500) as the broad-market control.

> ⚠️ Research only, not investment advice. Backtests are simplistic (no costs/slippage) — meant to
> surface hypotheses worth deeper validation, not production strategies.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install yfinance pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection

**TSA checkpoint volumes**: TSA's site publishes one page per calendar year (`/travel/passenger-volumes/{year}`)
plus a live current-year page at `/travel/passenger-volumes`, each containing a plain HTML
`Date, Numbers` table. We pull every year from 2019 (pre-COVID baseline) through the present and
concatenate. This is a small, infrequent set of GET requests against a public government page — we
add a short delay between requests and a descriptive User-Agent to be a good citizen.

**Stock data**: price + volume via `yfinance`, as in the companion notebooks.


In [ ]:
USER_AGENT = "flightDataResearch-notebook/1.0 (personal research; contact: bruno.inzunza24@gmail.com)"

def fetch_tsa_year(year_or_current):
    """Scrape one TSA passenger-volumes page (a year archive, or the live current-year page)
    into a Date -> Numbers Series."""
    slug = "" if year_or_current == "current" else f"/{year_or_current}"
    url = f"https://www.tsa.gov/travel/passenger-volumes{slug}"
    r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
    r.raise_for_status()
    html = r.text
    start = html.find("<table")
    end = html.find("</table>", start)
    table_html = html[start:end]
    rows = re.findall(
        r'<td class="text-align-center">([\d/]+)</td>\s*'
        r'<td class="text-align-center">([\d,]+)</td>',
        table_html,
    )
    if not rows:
        return pd.Series(dtype="float64")
    dates = pd.to_datetime([d for d, _ in rows], format="%m/%d/%Y")
    values = [float(v.replace(",", "")) for _, v in rows]
    return pd.Series(values, index=dates).sort_index()

EARLIEST_YEAR = 2019
current_year = pd.Timestamp.today().year

tsa_parts = []
for yr in range(EARLIEST_YEAR, current_year):
    tsa_parts.append(fetch_tsa_year(yr))
    time.sleep(0.3)
tsa_parts.append(fetch_tsa_year("current"))  # covers the current calendar year, updated daily

tsa = pd.concat(tsa_parts)
tsa = tsa[~tsa.index.duplicated(keep="last")].sort_index()
tsa.name = "tsa_checkpoint_volume"
print(f"TSA data: {tsa.index.min().date()} to {tsa.index.max().date()}, {len(tsa)} days")
tsa.tail()


In [ ]:
import yfinance as yf

TICKERS = {
    "DAL": "DAL",
    "UAL": "UAL",
    "AAL": "AAL",
    "LUV": "LUV",
    "JETS": "JETS",
    "SPX": "^GSPC",
}

stock_raw = yf.download(
    list(TICKERS.values()),
    start=tsa.index.min(),
    auto_adjust=True,
    progress=False,
    group_by="ticker",
)

stock = {}
for name, symbol in TICKERS.items():
    d = stock_raw[symbol][["Close", "Volume"]].rename(
        columns={"Close": f"{name}_close", "Volume": f"{name}_vol"}
    )
    stock[name] = d

stock_prices = pd.concat(stock.values(), axis=1)
stock_prices.index = pd.to_datetime(stock_prices.index)
stock_prices = stock_prices.sort_index()
stock_prices.tail()


## 2. Calendar Alignment & De-Seasonalizing

Two structural issues, both familiar from the companion notebooks:

1. **Calendar mismatch**: TSA reports every day of the year (including weekends/holidays); stocks
   trade ~252 days/year. We roll non-trading-day TSA volume forward into the next trading day, same
   approach as the Wikipedia-pageviews notebook.
2. **Day-of-week seasonality**: air travel is heavily seasonal by weekday (Fridays/Sundays are peak
   travel days, Tuesdays/Wednesdays are the trough) — raw day-over-day changes in TSA volume are
   dominated by this pattern, not by any underlying demand signal. The standard practice (used in the
   real 2020-2021 recovery trade) is to compare each day to the **same weekday one year earlier** —
   we compute that "year-over-year, same-weekday" growth rate as our primary de-seasonalized series.


In [ ]:
full = stock_prices.join(tsa, how="outer").sort_index()

trading_days = pd.Index(stock_prices.dropna(how="all").index)
tsa_full_sorted = full[["tsa_checkpoint_volume"]].sort_index()

bucket = pd.Series(pd.NaT, index=tsa_full_sorted.index, dtype="object")
is_trading_day = bucket.index.isin(trading_days)
bucket.loc[is_trading_day] = bucket.index[is_trading_day]
bucket = bucket.bfill()

tsa_rolled_fwd = tsa_full_sorted.groupby(bucket).sum(min_count=1)
tsa_rolled_fwd.index.name = "date"
tsa_rolled_fwd.columns = ["tsa_checkpoint_volume"]

df = stock_prices.join(tsa_rolled_fwd, how="inner").dropna(subset=[f"{t}_close" for t in TICKERS])

# Year-over-year, same-weekday growth: compare today's TSA volume to ~364 days ago (52 weeks,
# so the day-of-week lines up exactly) rather than 365 days ago.
tsa_yoy = tsa / tsa.shift(364) - 1
df["tsa_yoy_growth"] = tsa_yoy.reindex(df.index)
df.tail()


## 3. Overview: The COVID Natural Experiment

Plotting raw TSA volume and airline prices together should make the 2020 crash/recovery obvious —
a useful sanity check that the data pipeline is capturing something real before we lean on subtler
statistics.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

tsa.plot(ax=axes[0], title="TSA Daily Checkpoint Volume (raw, 7-day calendar)", color="tab:green")
axes[0].set_ylabel("Travelers screened / day")

norm = df[[f"{t}_close" for t in ["DAL", "UAL", "AAL", "LUV", "JETS"]]].dropna()
norm = norm / norm.iloc[0] * 100
norm.plot(ax=axes[1], title="Airline Stocks & JETS ETF (Indexed to 100)")
axes[1].set_ylabel("Indexed Price")

plt.tight_layout()
plt.show()


## 4. Returns & Correlation

We use the de-seasonalized YoY growth rate's *week-over-week change* (not its level) as the "signal"
series — the level is highly autocorrelated (recovery is a slow trend), so the interesting variation
for a trading signal is whether the pace of recovery is accelerating or decelerating.


In [ ]:
rets = pd.DataFrame(index=df.index)
for t in TICKERS:
    rets[f"{t}_ret"] = np.log(df[f"{t}_close"]).diff()

rets["tsa_yoy_chg"] = df["tsa_yoy_growth"].diff(5)  # ~1 trading week change in YoY growth pace
rets = rets.dropna(how="all")
rets.tail()


In [ ]:
corr_cols = [f"{t}_ret" for t in TICKERS] + ["tsa_yoy_chg"]
corr = rets[corr_cols].corr()

plt.figure(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation: Stock Returns vs. TSA YoY-Growth Pace Change")
plt.tight_layout()
plt.show()
corr["tsa_yoy_chg"].sort_values(ascending=False)


**Read this like the companion notebooks**: if airline names (DAL/UAL/AAL/LUV/JETS) correlate with
`tsa_yoy_chg` meaningfully more than SPX does, that's evidence the TSA signal is airline-specific
demand information, not just a proxy for general risk-on sentiment.


In [ ]:
window = 60
fig, ax = plt.subplots()
for t in ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]:
    roll = rets[f"{t}_ret"].rolling(window).corr(rets["tsa_yoy_chg"])
    roll.plot(ax=ax, label=t, alpha=0.8)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title(f"Rolling {window}d Correlation: Return vs. TSA YoY-Growth Pace Change")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Lead-Lag Analysis & Granger Causality

Does the stock market anticipate travel recovery before TSA data confirms it (price leads), or does
TSA data reveal real demand information the market hasn't fully priced in yet (TSA leads)? We test
both directions.


In [ ]:
def lagged_corr(y, x, max_lag=10):
    """Positive lag = x's past leads y's present."""
    return pd.Series({lag: y.corr(x.shift(lag)) for lag in range(-max_lag, max_lag + 1)})

max_lag = 10
fig, axes = plt.subplots(3, 2, figsize=(13, 12))
for ax, t in zip(axes.flat, ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]):
    sub = rets[[f"{t}_ret", "tsa_yoy_chg"]].dropna()
    ll = lagged_corr(sub[f"{t}_ret"], sub["tsa_yoy_chg"], max_lag)
    colors = ["#d62728" if l == 0 else "#1f77b4" for l in ll.index]
    ll.plot(kind="bar", ax=ax, color=colors)
    ax.set_title(f"{t}: TSA growth-pace (lag) vs return (t)")
    ax.set_xlabel("Lag (positive = TSA leads stock)")
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

for t in ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]:
    print(f"\n=== {t}: does TSA YoY growth-pace change Granger-cause return? ===")
    sub = rets[[f"{t}_ret", "tsa_yoy_chg"]].dropna()
    try:
        grangercausalitytests(sub, maxlag=3, verbose=True)
    except Exception as e:
        print(f"  skipped ({e})")


## 6. Event Study: Travel-Recovery Acceleration Shocks

Flag days where the pace of YoY travel recovery accelerates or decelerates sharply (z-score of the
week-over-week change in YoY growth), then look at forward airline returns. This is the direct analog
of the "attention shock" event study in the Wikipedia notebook, applied to a fundamentals-adjacent
signal instead of an attention proxy.


In [ ]:
def zscore(s, window=90):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

z = zscore(rets["tsa_yoy_chg"])
shock_threshold = 2.0
accel_days = z[z > shock_threshold].index
decel_days = z[z < -shock_threshold].index
print(f"Acceleration-shock days: {len(accel_days)}, Deceleration-shock days: {len(decel_days)}")

horizons = [1, 3, 5, 10, 20]
event_summary = {}
for t in ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]:
    fwd = {h: np.log(df[f"{t}_close"]).diff(h).shift(-h) for h in horizons}
    fwd_df = pd.DataFrame(fwd)
    event_summary[t] = pd.DataFrame({
        "after_acceleration": fwd_df.reindex(accel_days).mean(),
        "after_deceleration": fwd_df.reindex(decel_days).mean(),
    })

for t, tbl in event_summary.items():
    print(f"\n--- {t}: mean forward return ---")
    print(tbl.round(4))


## 7. Toy Backtest: Trade the Recovery-Pace Signal

Go long the JETS ETF (and, separately, a single-airline basket) when the TSA YoY growth-pace
z-score is positive (recovery accelerating), flat/short otherwise. No transaction costs modeled —
a raw-signal check, not a production backtest.


In [ ]:
def sharpe(returns, periods_per_year=252):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)

signal = np.sign(z.reindex(rets.index)).shift(1)  # trade on yesterday's signal, avoid lookahead

basket_ret = rets[[f"{t}_ret" for t in ["DAL", "UAL", "AAL", "LUV"]]].mean(axis=1)
strategies = pd.DataFrame({
    "buy_hold_JETS": rets["JETS_ret"],
    "signal_JETS": signal * rets["JETS_ret"],
    "buy_hold_airline_basket": basket_ret,
    "signal_airline_basket": signal * basket_ret,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: TSA Recovery-Pace Signal vs. Buy & Hold")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(252),
})


## 8. Robustness Check: Post-2022 Only (Excluding the COVID Shock)

Everything above spans 2019-present, and the 2020-2021 COVID crash/recovery is a single, enormous
natural experiment sitting inside that sample — it could easily be driving the entire
correlation/event-study/backtest story on its own. Here we re-run Sections 4-7 restricted to trading
days from **2022-01-01** onward, i.e. after the initial recovery had largely played out, to check
whether any of this holds in a more normal travel environment or was a COVID-era artifact.

All rolling statistics (`tsa_yoy_chg`, the acceleration/deceleration z-score, the backtest signal)
were already computed above using the *full* 2019-present history — that's correct and deliberate,
since they're purely backward-looking (no lookahead), so truncating the input history would just
create artificial NaN gaps at the start of 2022 for no benefit. What changes here is only which
**dates we evaluate/report** — we slice the already-computed series down to 2022-01-01 onward rather
than recomputing from scratch on a shorter window.


In [ ]:
POST_2022_START = "2022-01-01"

rets_post = rets.loc[POST_2022_START:]
df_post = df.loc[POST_2022_START:]

print(f"Restricted sample: {rets_post.index.min().date()} to {rets_post.index.max().date()}, "
      f"{len(rets_post)} trading days (full sample was {len(rets)})")


In [ ]:
corr_post = rets_post[corr_cols].corr()

plt.figure(figsize=(8, 7))
sns.heatmap(corr_post, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation (Post-2022 Only): Stock Returns vs. TSA YoY-Growth Pace Change")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "full_sample_2019+": corr["tsa_yoy_chg"],
    "post_2022_only": corr_post["tsa_yoy_chg"],
}).sort_values("post_2022_only", ascending=False)


In [ ]:
window = 60
fig, ax = plt.subplots()
for t in ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]:
    roll = rets_post[f"{t}_ret"].rolling(window).corr(rets_post["tsa_yoy_chg"])
    roll.plot(ax=ax, label=t, alpha=0.8)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title(f"Post-2022 Rolling {window}d Correlation: Return vs. TSA YoY-Growth Pace Change")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12))
for ax, t in zip(axes.flat, ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]):
    sub = rets_post[[f"{t}_ret", "tsa_yoy_chg"]].dropna()
    ll = lagged_corr(sub[f"{t}_ret"], sub["tsa_yoy_chg"], max_lag)
    colors = ["#d62728" if l == 0 else "#1f77b4" for l in ll.index]
    ll.plot(kind="bar", ax=ax, color=colors)
    ax.set_title(f"Post-2022 {t}: TSA growth-pace (lag) vs return (t)")
    ax.set_xlabel("Lag (positive = TSA leads stock)")
plt.tight_layout()
plt.show()


In [ ]:
for t in ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]:
    print(f"\n=== Post-2022 {t}: does TSA YoY growth-pace change Granger-cause return? ===")
    sub = rets_post[[f"{t}_ret", "tsa_yoy_chg"]].dropna()
    try:
        grangercausalitytests(sub, maxlag=3, verbose=True)
    except Exception as e:
        print(f"  skipped ({e})")


In [ ]:
accel_days_post = accel_days[accel_days >= POST_2022_START]
decel_days_post = decel_days[decel_days >= POST_2022_START]
print(f"Post-2022 acceleration-shock days: {len(accel_days_post)}, "
      f"deceleration-shock days: {len(decel_days_post)}")

event_summary_post = {}
for t in ["DAL", "UAL", "AAL", "LUV", "JETS", "SPX"]:
    fwd = {h: np.log(df[f"{t}_close"]).diff(h).shift(-h) for h in horizons}
    fwd_df = pd.DataFrame(fwd)
    event_summary_post[t] = pd.DataFrame({
        "after_acceleration": fwd_df.reindex(accel_days_post).mean(),
        "after_deceleration": fwd_df.reindex(decel_days_post).mean(),
    })

for t, tbl in event_summary_post.items():
    print(f"\n--- Post-2022 {t}: mean forward return ---")
    print(tbl.round(4))


In [ ]:
strategies_post = strategies.loc[POST_2022_START:]
cum_post = (1 + strategies_post.fillna(0)).cumprod()  # re-based to 1.0 at the start of this window
cum_post.plot(title="Cumulative Return (Post-2022 Only): TSA Recovery-Pace Signal vs. Buy & Hold")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "annualized_sharpe": strategies_post.apply(sharpe),
    "total_return": cum_post.iloc[-1] - 1,
    "annualized_vol": strategies_post.std() * np.sqrt(252),
})


## 9. Takeaways & Next Steps

- **COVID-era vs. post-2022 comparison (Section 8)**: look at whether the correlation, event-study,
  and backtest results held up, weakened, or reversed once the COVID natural experiment is excluded.
  A signal that only exists 2019-2021 is a COVID-recovery artifact, not a durable edge — treat the
  post-2022 numbers as the more honest read of whether this generalizes.
- **Compare airlines to each other, not just to SPX**: LUV is almost entirely domestic leisure
  travel; DAL/UAL/AAL have larger international/business mixes. If TSA (a domestic-checkpoint-only
  metric) correlates much more strongly with LUV than with DAL, that's a useful clue about *why* the
  signal works, not just *that* it works.
- **Granger causality both ways matters**: if the stock leads TSA data (not the reverse), that
  reframes TSA data as a confirmation/validation signal for a thesis already priced in, rather than a
  standalone leading indicator — still useful, just used differently (position sizing/conviction, not
  entry timing).
- **Extend the dataset**: TSA is U.S. domestic-checkpoint-only. For international carriers or a
  global travel thesis, pair this with IATA/ICAO traffic statistics, airport-specific data (e.g. BTS
  T-100 segment data), or ADS-B-based flight-count feeds (e.g. OpenSky Network) for actual global
  flight volumes rather than a US-checkpoint proxy.
- **Robustness**: as with the companion notebooks, validate out-of-sample and with transaction-
  cost-adjusted backtests before treating any of this as an actual edge.
